In [1]:
import os 
os.chdir('../../..')

In [2]:

import json
import pandas as pd
import plotly.graph_objects as go

from pathlib import Path
from IPython.display import display, Markdown
from plotly.subplots import make_subplots

In [5]:
report_path = Path('results/cluster_reports/consolidated_report_materials.json')
with report_path.open() as f:
    report_data = json.load(f)


In [6]:
metrics = ['structure_score', 'property_score', 'structure_class_score', 'separation_score', 'category_score']
plot_data = []

for key, val in report_data.items():
    # Only process the files that contain cluster scores
    if key.endswith('_cluster_scores.json'):
        # Create a display name (e.g. "kmeans_soap_embedding")
        method = key.split('/')[0].replace('materials_', '')
        embedding = val['embedding']
        name = f"{method}_{embedding}"
        
        scores = val['molecular_cluster_score']
        
        # Append to our list
        plot_data.append({
            'Name': name,
            'structure_score': scores.get('structure_score', 0),
            'property_score': scores.get('property_score', 0),
            'structure_class_score': scores.get('structure_class_score', 0),
            'separation_score': scores.get('separation_score', 0),
            'category_score': scores.get('category_score', 0)
        })

df = pd.DataFrame(plot_data)

# 3. Create the Radar Chart
fig = go.Figure()

for index, row in df.iterrows():
    fig.add_trace(go.Scatterpolar(
        # We append the first metric to the end to close the circle on the radar chart
        r=[row[m] for m in metrics] + [row[metrics[0]]],
        theta=metrics + [metrics[0]],
        mode='lines+markers',
        name=row['Name'],
        line=dict(width=2),
        marker=dict(size=6)
    ))

# 4. Apply Dark Theme and Formatting (matching the screenshot)
fig.update_layout(
    title=dict(
        text="Materials Embeddings Performance",
        x=0.5,
        font=dict(size=20)
    ),
    template="plotly_dark",
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1],
            showline=False,
            gridcolor='#444',
            tickfont=dict(size=12, color='#eee')
        ),
        angularaxis=dict(
            tickfont=dict(size=14, color='#eee'),
            gridcolor='#444'
        ),
        bgcolor='#111'
    ),
    paper_bgcolor='#111',  # Outer background
    showlegend=True,
    legend=dict(
        x=1.1,
        y=0.9,
        font=dict(size=12, color='#eee')
    )
)

fig.show()